# Обучение и квантование Qwen2.5-1.5B (QLoRA -> GGUF Q4_K_M) для Infinix Note 30

Этот блокнот выполняет полный цикл:
1. Установка окружения (PyTorch, Transformers, PEFT, BitsAndBytes)
2. Подготовка русскоязычного датасета заметок в формате ChatML
3. Обучение адаптера LoRA на 4-битной модели (QLoRA) с маскированием промпта
4. Слияние весов LoRA с базовой моделью
5. Конвертация в GGUF и квантование в `Q4_K_M` через `llama.cpp`
6. Скачивание готового файла `qwen2.5-1.5b-notes-q4_k_m.gguf` для Android-приложения

In [ ]:
# 1. Проверка GPU (достаточно бесплатной Tesla T4 в Colab)
!nvidia-smi

In [ ]:
# 2. Установка зависимостей
!pip install -q -U torch torchvision torchaudio
!pip install -q transformers>=4.44.0 peft>=0.12.0 datasets bitsandbytes accelerate trl sentencepiece tiktoken

In [ ]:
# 3. Подготовка обучающих данных (ChatML)
import json

sample_data = [
    {
        "messages": [
            {"role": "system", "content": "Ты — умный и лаконичный русскоязычный офлайн-ассистент для смартфона Infinix Note 30. Помогай с заметками, вопросами и пересказами."},
            {"role": "user", "content": "Сделай краткое резюме заметки:\n\nКупить продукты на неделю: молоко, хлопья, кофе. В 14:00 созвон с командой по архитектуре LLM. Проверить температуру процессора Helio G99 при инференсе."},
            {"role": "assistant", "content": "Краткое резюме:\n1. Покупки: молоко, хлопья, кофе.\n2. Встреча: 14:00 — созвон по архитектуре LLM.\n3. Тестирование: замер температуры Helio G99 при генерации."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Ты — умный и лаконичный русскоязычный офлайн-ассистент для смартфона Infinix Note 30. Помогай с заметками, вопросами и пересказами."},
            {"role": "user", "content": "Выдели список задач (Action Items) из заметки:\n\nОбсудили релиз v1.0.0. Нужно собрать релизный APK через GitHub Actions, протестировать на телефоне и настроить порог потоков на 2 ядра Cortex-A76."},
            {"role": "assistant", "content": "Список задач:\n- [ ] Собрать релизный APK через GitHub Actions\n- [ ] Протестировать работу приложения на смартфоне\n- [ ] Установить лимит инференса на 2 потока (Cortex-A76)"}
        ]
    }
]

# Размножаем или загружаем свои заметки
with open("train.jsonl", "w", encoding="utf-8") as f:
    for item in sample_data * 50:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Данные подготовлены: train.jsonl")

In [ ]:
# 4. Скрипт файнтюнинга QLoRA
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "qwen2.5-1.5b-notes-lora"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)

def tokenize_fn(batch):
    texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]
    enc = tokenizer(texts, truncation=True, max_length=1536)
    enc["labels"] = enc["input_ids"].copy()
    return enc

ds = load_dataset("json", data_files={"train": "train.jsonl"})
tokenized_ds = ds.map(tokenize_fn, batched=True, remove_columns=["messages"])

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="no",
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True)
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Обучение LoRA завершено!")

In [ ]:
# 5. Слияние LoRA с базовой моделью
from peft import PeftModel

MERGED_DIR = "qwen2.5-1.5b-notes-merged"
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True)
model_merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
model_merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Модель успешно объединена в:", MERGED_DIR)

In [ ]:
# 6. Сборка llama.cpp и конвертация в GGUF Q4_K_M
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!cmake -B llama.cpp/build -S llama.cpp -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --config Release -j $(nproc) --target llama-quantize llama-cli
!pip install -q -r llama.cpp/requirements.txt

# Конвертация Safetensors -> GGUF FP16
!python3 llama.cpp/convert_hf_to_gguf.py qwen2.5-1.5b-notes-merged --outfile model-f16.gguf --outtype f16

# Квантование FP16 -> Q4_K_M
!./llama.cpp/build/bin/llama-quantize model-f16.gguf qwen2.5-1.5b-notes-q4_k_m.gguf Q4_K_M

!ls -lh qwen2.5-1.5b-notes-q4_k_m.gguf

In [ ]:
# 7. Скачивание готового GGUF файла модели на локальный компьютер
from google.colab import files
files.download("qwen2.5-1.5b-notes-q4_k_m.gguf")